# Kigurumi Head Reference Agent (v1)

End-to-end Agent that turns a target character + expression reference into:

1. **Step 1** — unobstructed head four-view (general prompt + ≤2 targeted refines).
2. **Dataset pick** — 0–2 visually similar finished shop products from `dataset/`.
3. **Step 2** — kigurumi head shell four-view (general + ≤2 refines).
4. **Step 3** — product-photo four-view (single pass + drift review).

**Models**
- Image: `gpt-image-2` via OpenAI-compatible `images.edit`.
- Reasoning / vision: Qwen3.5-VL (`qwen-vl-max-latest`) via Aliyun Dashscope OpenAI-compatible endpoint.

**Setup** (run from the repo root):
```bash
pip install -r notebook/requirements.txt
cp notebook/.env.example notebook/.env   # then fill in IMAGE_API_KEY and REASON_API_KEY
jupyter notebook kigurumi_agent.ipynb
```

Outputs land in `outputs/<timestamp>_<label>/` — every iteration's candidates plus a `run.json` audit trail.

In [1]:
import sys, importlib
from pathlib import Path

# Allow imports of the agent modules that live under ./notebook/
_module_dir = Path.cwd() / 'notebook'
if str(_module_dir) not in sys.path:
    sys.path.insert(0, str(_module_dir))

import prompts, notebook.config, notebook.agent
importlib.reload(prompts); importlib.reload(notebook.config); importlib.reload(notebook.agent)
from notebook.prompts import load_all
from notebook.config import load_config, DATASET_DIR, OUTPUT_DIR
from notebook.agent import KigurumiAgent

for step_id, sp in load_all().items():
    print(f'step{step_id}: general={len(sp.general)} chars, refines={sp.refinement_keys()}')

cfg = load_config()
print('Image :', cfg.image.model, '@', cfg.image.base_url)
print('Reason:', cfg.reason.model, '@', cfg.reason.base_url)
print('Image API Key:', cfg.image.api_key[:4] + '...' if cfg.image.api_key else None)
print('Reason API Key:', cfg.reason.api_key[:4] + '...' if cfg.reason.api_key else None)
print('Dataset folders:', len(list(DATASET_DIR.iterdir())))
print('Output dir     :', OUTPUT_DIR)
print('image size      :', cfg.image.size)
print('image quality   :', cfg.image.quality)
print('image n         :', cfg.image.n)

step1: general=429 chars, refines=['expression', 'face', 'childlike']
step2: general=802 chars, refines=['shell', 'too3d', 'expression', 'face']
step3: general=474 chars, refines=['four', 'eight']
Image : gpt-image-2-2026-04-21 @ https://api.openai.com/v1
Reason: qwen3.6-plus @ https://dashscope.aliyuncs.com/compatible-mode/v1
Image API Key: sk-p...
Reason API Key: sk-4...
Dataset folders: 82
Output dir     : C:\Project\ai-kigurumi-head-reference\outputs
image size      : 1024x1024
image quality   : high
image n         : 1


## Inputs

All paths below are resolved from the repo root. Drop your reference images into `references/ganyu/` (already gitignored).

- `CHAR_REF_PATHS`: 1–4 official / structure references for the target character.
- `CHAR_INFO`: a short text block: IP, character name, key visual traits — used by the dataset picker and reviewer.
- `EXPR_REF_PATH`: one expression reference image.

In [2]:
CHAR_REF_PATHS = [
    'references/Niko/front.png',
]
CHAR_INFO = '''蔚蓝档案-妮可，粉色短发带狐狸耳朵的少女，表情是张嘴坏坏的笑'''
EXPR_REF_PATH = 'references/Niko/exp.png'
RUN_LABEL = 'Niko-demo'

In [3]:
agent_runner = KigurumiAgent(cfg)
result = agent_runner.run(
    char_ref_paths=CHAR_REF_PATHS,
    char_info=CHAR_INFO,
    expr_ref_path=EXPR_REF_PATH,
    run_label=RUN_LABEL,
)
print('Run dir:', result.run_dir)
print('Step 1 best:', result.step1.best.path)
print('Step 2 best:', result.step2.best.path)
print('Step 3 best:', result.step3.best.path)
print('Similar picks:', [p.name for p in result.similar])

[21:42:33] [agent] run start: run_dir=20260515-214233_Niko-demo, char_refs=1, expr_ref=exp.png, image_model=gpt-image-2-2026-04-21, reason_model=qwen3.6-plus, max_refine=2
[21:42:33] [step1] start: base_refs=2, max_refine=2
[21:42:33] [step1] iter0 generate (key=general)
[21:45:04] [step1] iter0 generated 1 candidates in 150.5s
[21:45:04] [step1] iter0 review
[21:47:14] [step1] iter0 verdict=refine best=0 refine_key=expression (129.7s) notes='[B] 表情错误，当前为闭嘴微笑，需改为张嘴坏笑（参考表情图）；[A] 缺失头顶蓝色光环'
[21:48:09] [refine-adjust] mode=rewrite, key=expression->expression, reason='评审备注包含新增装饰物（头顶蓝色光环），该维度超出所有预写prompt覆盖范围，且表情需具体化，符合rewrite模式条件。'
[21:48:09] [step1] iter1 generate (key=expression, mode=rewrite, primary=iter00_general_0.png)
[21:51:00] [step1] iter1 generated 1 candidates in 171.1s
[21:53:25] [step1] iter1 verdict=refine best=0 refine_key=expression (145.6s) notes='[B] 眼神过于圆润清澈，未还原目标表情参考图中的半眯/坏笑神态；嘴形虽张开但缺乏“坏坏”的感觉'
[21:54:18] [refine-adjust] mode=append, key=expression->expression, reason='备注

## Inspect results inline

In [ ]:
from IPython.display import Image, display, Markdown

def show(label, path):
    display(Markdown(f'### {label}\n`{path}`'))
    display(Image(filename=str(path)))

show('Step 1 — unobstructed head turnaround', result.step1.best.path)
show('Step 2 — kigurumi shell turnaround', result.step2.best.path)
show('Step 3 — product-photo four-view', result.step3.best.path)

if result.similar:
    display(Markdown('### Similar shop products used as references'))
    for p in result.similar:
        show(p.stem, p)

## Audit trail

Every candidate and verdict is saved. `run.json` shows what was generated each iteration, what the reviewer said, and which refine prompt was picked.

In [ ]:
import json
audit = json.loads((result.run_dir / 'run.json').read_text(encoding='utf-8'))
for step in ('step1', 'step2', 'step3'):
    print(f'--- {step} ---')
    for it in audit[step]['iterations']:
        v = it.get('verdict', {})
        print(f"  {it['prompt_key']}: best={v.get('best_index')} verdict={v.get('verdict')} "
              f"refine={v.get('refine_key')} notes={v.get('notes')}")